## Setup

Reads parquet files for EES download metrics from the Volume root and writes them incrementally to Delta tables in Unity Catalog.

These download file types (zip, csv, table-tool, permalink-table) have only ever been written flat to the Volume root by the upstream Data Factory pipeline - there are no structured subfolders for them.

### Prerequisites

- **Cluster libraries**: `sparklyr` and `arrow` are pre-installed on DBR. `duckdb` and `DBI` are installed at runtime if missing.
- **Permissions**: Read access to the source Volume at `/Volumes/catalog_40_copper_statistics_services/statistics_services/mv_statistics_services/`. Write access to `catalog_40_copper_statistics_services.statistics_services`.
- **Filename format**: Source parquet files must be prefixed with `YYYYMMdd-HHmmss` (e.g., `20260216-170202_public-zip-downloads.parquet`). This prefix is parsed into `_file_datetime` and `_file_date` metadata columns.

In [0]:
%r
# -- Configuration -----------------------------------------------------------

TARGET_CATALOG <- "catalog_40_copper_statistics_services"
TARGET_SCHEMA  <- "statistics_services"

# Volume root where download parquet files are written flat by Data Factory.
# These file types have never had structured subfolders.
volume_root <- "/Volumes/catalog_40_copper_statistics_services/statistics_services/mv_statistics_services/"
volume_path <- paste0(volume_root, "public-api/")

# -- Dependencies ------------------------------------------------------------

options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/latest"))

if (!requireNamespace("duckdb", quietly = TRUE)) {
    install.packages("duckdb")
}
if (!requireNamespace("DBI", quietly = TRUE)) {
    install.packages("DBI")
}

library(sparklyr)
library(dplyr)
library(duckdb)
library(DBI)
library(arrow)
sc <- spark_connect(method = "databricks")

In [0]:
%r
#' Write new parquet files incrementally to a Delta table
#'
#' Supports dual-source file discovery and basename-level deduplication.
#' See raw_ees_api_queries for full documentation.
#'
#' @param folder Character. Subfolder name relative to the base Volume path.
#' @param pattern Character or NULL. Regex pattern to filter filenames.
#' @param table_name Character. Target table name (without catalog/schema prefix).
#' @param root_pattern Character or NULL. Regex pattern for Volume root files.
#' @param base_path Character or NULL. Base Volume path. Defaults to volume_path.
write_to_delta_incremental <- function(
    folder,
    pattern = NULL,
    table_name,
    root_pattern = NULL,
    base_path = volume_path
) {
    full_table_name <- paste0(TARGET_CATALOG, ".", TARGET_SCHEMA, ".", table_name)

    full_folder <- if (!is.null(base_path)) {
        gsub("//+", "/", paste0(base_path, "/", folder, "/"))
    } else {
        sub("/?$", "/", folder)
    }

    subfolder_files <- if (dir.exists(full_folder)) {
        raw <- list.files(full_folder, recursive = FALSE)
        full_paths <- paste0(full_folder, raw)
        if (!is.null(pattern)) full_paths[grepl(pattern, full_paths)] else full_paths
    } else {
        character(0)
    }

    root_files <- if (!is.null(root_pattern)) {
        raw <- list.files(volume_root, recursive = FALSE)
        full_paths <- paste0(volume_root, raw[grepl(root_pattern, raw)])
        full_paths[grepl("\\.parquet$", full_paths)]
    } else {
        character(0)
    }

    combined <- c(subfolder_files, root_files)
    all_files <- combined[!duplicated(basename(combined))]

    if (length(subfolder_files) > 0 && length(root_files) > 0) {
        n_deduped <- length(combined) - length(all_files)
        message(sprintf(
            "Found files in subfolder (%d) + Volume root (%d) for %s%s",
            length(subfolder_files), length(root_files), table_name,
            if (n_deduped > 0) sprintf(" (%d duplicates removed)", n_deduped) else ""
        ))
    }

    if (length(all_files) == 0) {
        stop(sprintf("No source files found in %s or Volume root (pattern: %s / %s)",
                     full_folder,
                     if (is.null(pattern)) "all" else pattern,
                     if (is.null(root_pattern)) "none" else root_pattern))
    }

    # Sort by basename, not full path, to get the truly newest file.
    # Full-path sorting breaks with mixed directories (p > 2 in ASCII).
    # Spark SQL regexp_extract requires 3 args: (str, regexp, groupIdx).
    newest_file <- all_files[order(basename(all_files), decreasing = TRUE)[1]]
    newest_basename <- basename(newest_file)
    newest_in_table <- tryCatch({
        result <- sdf_sql(sc, sprintf(
            "SELECT 1 FROM %s WHERE regexp_extract(_source_file, '([^/]+)$', 1) = '%s' LIMIT 1",
            full_table_name, gsub("'", "''", newest_basename)
        )) %>% collect()
        nrow(result) > 0
    }, error = function(e) {
        msg <- conditionMessage(e)
        if (grepl("TABLE_OR_VIEW_NOT_FOUND|Table or view not found|AnalysisException", msg)) {
            NA
        } else {
            stop(e)
        }
    })

    if (isTRUE(newest_in_table)) {
        message(
            "No new files to process for ", table_name,
            " (", length(all_files), " files, newest already processed)"
        )
        return(invisible(list(
            table_name        = table_name,
            full_table_name   = full_table_name,
            status            = "skipped",
            source_file_count = length(all_files)
        )))
    }

    if (is.na(newest_in_table)) {
        message("Table doesn't exist yet, will create it.")
        new_files <- all_files
        pre_file_count <- 0L
    } else {
        all_files_df <- data.frame(
            `_source_file` = all_files,
            `_basename` = basename(all_files),
            stringsAsFactors = FALSE,
            check.names = FALSE
        )
        all_files_sdf <- copy_to(
            sc, all_files_df,
            name = paste0("temp_all_files_", table_name),
            overwrite = TRUE
        )
        existing_sdf <- sdf_sql(sc, sprintf(
            "SELECT DISTINCT regexp_extract(_source_file, '([^/]+)$', 1) AS _basename FROM %s",
            full_table_name
        ))
        new_files <- anti_join(all_files_sdf, existing_sdf, by = "_basename") %>%
            collect() %>% `[[`("_source_file")
        pre_file_count <- sdf_sql(sc, sprintf(
            "SELECT COUNT(DISTINCT _source_file) AS cnt FROM %s", full_table_name
        )) %>% collect() %>% `[[`("cnt")
    }

    if (length(new_files) == 0) {
        message("No new files to process for ", table_name)
        return(invisible(list(
            table_name        = table_name,
            full_table_name   = full_table_name,
            status            = "skipped",
            source_file_count = length(all_files)
        )))
    }

    message(
        "Processing ", length(new_files), " new files (",
        pre_file_count, " already processed)..."
    )

    file_list <- paste0("[", paste0("'", new_files, "'", collapse = ", "), "]")

    con <- DBI::dbConnect(duckdb::duckdb())
    on.exit(DBI::dbDisconnect(con, shutdown = TRUE))

    query <- sprintf(
        "SELECT
            *,
            filename AS _source_file,
            strptime(
                substr(regexp_extract(filename, '[^/]+$'), 1, 15),
                '%%Y%%m%%d-%%H%%M%%S'
            ) AS _file_datetime,
            strptime(
                substr(regexp_extract(filename, '[^/]+$'), 1, 8),
                '%%Y%%m%%d'
            )::DATE AS _file_date
        FROM read_parquet(%s, filename = true, union_by_name = true)",
        file_list
    )

    new_data <- DBI::dbGetQuery(con, query)

    if (nrow(new_data) == 0) {
        stop(sprintf(
            "New parquet files contained zero rows for %s (%d files)",
            table_name, length(new_files)
        ))
    }

    null_source   <- sum(is.na(new_data[["_source_file"]]))
    null_datetime <- sum(is.na(new_data[["_file_datetime"]]))
    null_date     <- sum(is.na(new_data[["_file_date"]]))

    if (null_source + null_datetime + null_date > 0) {
        stop(sprintf(
            paste0(
                "Metadata NULL check failed for %s (%d rows): ",
                "_source_file=%d, _file_datetime=%d, _file_date=%d NULLs"
            ),
            table_name, nrow(new_data), null_source, null_datetime, null_date
        ))
    }

    file_dates <- as.Date(new_data[["_file_date"]])

    if (any(file_dates > Sys.Date() + 1)) {
        stop(sprintf(
            "Future dates detected in %s: max _file_date is %s (today is %s)",
            table_name, max(file_dates), Sys.Date()
        ))
    }
    if (any(file_dates < as.Date("2020-01-01"))) {
        stop(sprintf(
            "Implausibly old dates in %s: min _file_date is %s",
            table_name, min(file_dates)
        ))
    }

    if (!is.na(newest_in_table)) {
        existing_cols <- tryCatch({
            colnames(sdf_sql(sc, sprintf("SELECT * FROM %s LIMIT 0", full_table_name)))
        }, error = function(e) character(0))

        if (length(existing_cols) > 0) {
            new_cols <- names(new_data)
            missing_cols <- setdiff(existing_cols, new_cols)
            extra_cols <- setdiff(new_cols, existing_cols)

            if (length(missing_cols) > 0) {
                stop(sprintf(
                    "Schema mismatch for %s: existing columns missing from new data: %s",
                    table_name, paste(missing_cols, collapse = ", ")
                ))
            }
            if (length(extra_cols) > 0) {
                stop(sprintf(
                    "Schema mismatch for %s: new columns not in existing table: %s. Resolve via schema evolution or manual ALTER TABLE.",
                    table_name, paste(extra_cols, collapse = ", ")
                ))
            }
        }
    }

    message("Writing ", nrow(new_data), " rows to ", full_table_name, "...")

    write_result <- tryCatch({
        pre_write_count <- if (!is.na(newest_in_table)) {
            sdf_sql(sc, sprintf(
                "SELECT COUNT(*) AS cnt FROM %s", full_table_name
            )) %>% collect() %>% `[[`("cnt")
        } else {
            0L
        }

        temp_sdf <- copy_to(
            sc, new_data,
            name = paste0("temp_", table_name),
            overwrite = TRUE,
            serializer = "arrow"
        )

        write_mode <- if (is.na(newest_in_table)) "overwrite" else "append"
        spark_write_table(
            temp_sdf,
            name = full_table_name,
            mode = write_mode
        )

        post_write_count <- sdf_sql(sc, sprintf(
            "SELECT COUNT(*) AS cnt FROM %s", full_table_name
        )) %>% collect() %>% `[[`("cnt")

        expected_count <- pre_write_count + nrow(new_data)
        if (post_write_count != expected_count) {
            stop(sprintf(
                "Post-write row count mismatch for %s: expected %d (pre=%d + new=%d), found %d",
                table_name, expected_count, pre_write_count, nrow(new_data), post_write_count
            ))
        }

        total_files <- pre_file_count + length(new_files)
        latest_file_date <- max(file_dates)

        message(sprintf(
            "Done! %s: %d rows written, %d total from %d files (latest: %s). Row count verified.",
            table_name, nrow(new_data), post_write_count, total_files, latest_file_date
        ))

        list(
            table_name        = table_name,
            full_table_name   = full_table_name,
            status            = "success",
            files_new         = length(new_files),
            files_total       = total_files,
            rows_written      = nrow(new_data),
            rows_total        = post_write_count,
            latest_file_date  = latest_file_date,
            source_file_count = length(all_files)
        )
    }, error = function(e) {
        message(sprintf("ERROR writing %s: %s", table_name, conditionMessage(e)))
        list(
            table_name        = table_name,
            full_table_name   = full_table_name,
            status            = "error",
            error_message     = conditionMessage(e),
            source_file_count = length(all_files)
        )
    })

    invisible(write_result)
}

## Write to Delta Tables (Incremental)

The cells below write each download dataset to Delta tables in `catalog_40_copper_statistics_services.statistics_services`.

On first run, all files are processed. On subsequent runs, only new files are appended.

These file types only exist at the Volume root (no structured subfolders).

| Table | Root pattern | Available since |
| --- | --- | --- |
| `raw_ees_zip_downloads` | public-zip-downloads | 2026-02-16 |
| `raw_ees_csv_downloads` | public-csv-downloads | 2026-02-16 |
| `raw_ees_table_tool_downloads` | public-table-tool-downloads | 2026-02-18 |
| `raw_ees_permalink_table_downloads` | public-permalink-table-downloads | 2026-02-25 |

In [0]:
%r
# Collect structured results from each table write for the final validation
# summary. Each call to write_to_delta_incremental() returns a named list with
# status, file counts, row counts, and latest file date.
audit_results <- list()

In [0]:
%r
audit_results[["raw_ees_zip_downloads"]] <- write_to_delta_incremental(
    folder = "zip-downloads",
    table_name = "raw_ees_zip_downloads",
    root_pattern = "public-zip-downloads"
)

In [0]:
%r
audit_results[["raw_ees_csv_downloads"]] <- write_to_delta_incremental(
    folder = "csv-downloads",
    table_name = "raw_ees_csv_downloads",
    root_pattern = "public-csv-downloads"
)

In [0]:
%r
audit_results[["raw_ees_table_tool_downloads"]] <- write_to_delta_incremental(
    folder = "table-tool-downloads",
    table_name = "raw_ees_table_tool_downloads",
    root_pattern = "public-table-tool-downloads"
)

In [0]:
%r
audit_results[["raw_ees_permalink_table_downloads"]] <- write_to_delta_incremental(
    folder = "permalink-table-downloads",
    table_name = "raw_ees_permalink_table_downloads",
    root_pattern = "public-permalink-table-downloads"
)

## Post-Ingestion Validation

Checks run after all tables are written:

1. **Audit summary** - displays a table of per-table status, file counts, row counts, and latest file dates
2. **Data freshness** - warns if the latest `_file_date` in any table is older than the threshold (default: 5 days)
3. **Source file count** - warns if the Delta table references more files than currently exist in the Volume (indicates source files were deleted)

In [0]:
%r
# -- Build audit summary from collected results --------------------------------
safe_get <- function(x, field, default = NA) {
    val <- x[[field]]
    if (is.null(val)) default else val
}

get_table_latest_date <- function(full_table_name) {
    tryCatch({
        sdf_sql(sc, sprintf(
            "SELECT MAX(_file_date) AS max_date FROM %s", full_table_name
        )) %>% collect() %>% `[[`("max_date") %>% as.character()
    }, error = function(e) NA_character_)
}

summary_df <- do.call(rbind, lapply(audit_results, function(r) {
    latest <- safe_get(r, "latest_file_date")
    latest <- if (is.null(latest) || is.na(latest)) {
        get_table_latest_date(r$full_table_name)
    } else {
        as.character(latest)
    }

    data.frame(
        table            = safe_get(r, "table_name"),
        status           = safe_get(r, "status"),
        files_new        = safe_get(r, "files_new", NA_integer_),
        files_total      = safe_get(r, "files_total", NA_integer_),
        source_files     = safe_get(r, "source_file_count", NA_integer_),
        rows_written     = safe_get(r, "rows_written", NA_integer_),
        rows_total       = safe_get(r, "rows_total", NA_integer_),
        latest_file_date = latest,
        stringsAsFactors = FALSE
    )
}))

cat("\n=== Ingestion Audit Summary ===", "\n")
cat("Run at:", format(Sys.time(), "%Y-%m-%d %H:%M:%S %Z"), "\n\n")
print(summary_df, row.names = FALSE)

# -- Data freshness: latest file date should be within threshold ---------------
FRESHNESS_THRESHOLD_DAYS <- 5
freshness_cutoff <- Sys.Date() - FRESHNESS_THRESHOLD_DAYS
stale_tables <- character(0)

for (i in seq_len(nrow(summary_df))) {
    latest <- summary_df$latest_file_date[i]
    if (!is.na(latest) && as.Date(latest) < freshness_cutoff) {
        stale_tables <- c(stale_tables, sprintf(
            "  %s (latest: %s)", summary_df$table[i], latest
        ))
    }
}

if (length(stale_tables) > 0) {
    warning(sprintf(
        "STALE DATA (threshold: %d days, cutoff: %s):\n%s",
        FRESHNESS_THRESHOLD_DAYS, freshness_cutoff,
        paste(stale_tables, collapse = "\n")
    ))
}

# -- Source vs table file count: detect deleted Volume files -------------------
file_count_issues <- character(0)

for (r in audit_results) {
    if (!is.null(r$source_file_count) && !is.null(r$full_table_name)) {
        table_file_count <- tryCatch({
            sdf_sql(sc, sprintf(
                "SELECT COUNT(DISTINCT _source_file) AS cnt FROM %s",
                r$full_table_name
            )) %>% collect() %>% `[[`("cnt")
        }, error = function(e) NA)

        if (!is.na(table_file_count) && table_file_count > r$source_file_count) {
            file_count_issues <- c(file_count_issues, sprintf(
                "  %s: %d files in table vs %d in Volume",
                r$table_name, table_file_count, r$source_file_count
            ))
        }
    }
}

if (length(file_count_issues) > 0) {
    warning(sprintf(
        "SOURCE FILE COUNT MISMATCH (files may have been deleted from Volume):\n%s",
        paste(file_count_issues, collapse = "\n")
    ))
}

cat("\n=== Validation complete ===", "\n")